# Deep learning on images: telling A, B and C apart

This short notebook trains a small convolutional neural network (CNN) to recognise the letters **A**, **B** and **C**, even when they are scaled, sheared, rotated, shifted and bent by a random vector field.

**Parts**
1. Draw the letters
2. Distort them randomly (affine transform + elastic vector field)
3. Build training, validation and test sets
4. Define a CNN
5. Train it
6. Evaluate: accuracy, confusion matrix, mistakes
7. Robustness: what happens with stronger distortions than seen in training?
8. Look inside: filters, feature maps and saliency

Everything needed (PyTorch, SciPy, Matplotlib) is preinstalled in Google Colab. A GPU is optional: training takes about a minute on a CPU. (In Colab: *Runtime → Change runtime type → GPU* if you want one.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates, gaussian_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

## Part 1: Draw the letters

Each letter is rendered as a 32 × 32 grey-scale image in three different fonts that ship with Matplotlib. Using several fonts makes the task harder and forces the network to learn the *shape* of a letter rather than one particular drawing of it.

In [ ]:
SIZE = 32
LETTERS = ["A", "B", "C"]
FONTS = ["DejaVu Sans", "DejaVu Serif", "DejaVu Sans Mono"]

def render_letter(letter, font="DejaVu Sans", size=SIZE):
    """Draw a letter as a white-on-black float image of shape (size, size)."""
    fig = plt.figure(figsize=(1, 1), dpi=size)
    fig.text(0.5, 0.5, letter, fontsize=50, family=font, ha="center", va="center")
    fig.canvas.draw()
    img = 1 - np.asarray(fig.canvas.buffer_rgba())[..., 0] / 255.0
    plt.close(fig)
    return img.astype(np.float32)

templates = {(l, f): render_letter(l, f) for l in LETTERS for f in FONTS}

fig, axes = plt.subplots(len(LETTERS), len(FONTS), figsize=(6, 6))
for ax, ((letter, font), img) in zip(axes.flat, templates.items()):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{letter}  ({font})", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 2: Random distortions

Every training image is a template that has been
* **scaled** (overall size and aspect ratio),
* **sheared** and **rotated**,
* **shifted** by a few pixels,
* **bent** by a smooth random vector field (an *elastic distortion*, a classic trick from handwriting recognition), and
* given a little noise.

All distortions are combined into one coordinate map and applied in a single interpolation step with `map_coordinates`, the same backward-warping approach as in the lens experiments. The parameter `strength` scales every distortion together: 0 means none, 1 is the setting used for training.

In [ ]:
def random_field(size, max_disp, smooth, rng):
    """Smooth random vector field; its longest vector is max_disp pixels."""
    dy = gaussian_filter(rng.standard_normal((size, size)), smooth)
    dx = gaussian_filter(rng.standard_normal((size, size)), smooth)
    norm = np.hypot(dy, dx).max() + 1e-12
    return dy / norm * max_disp, dx / norm * max_disp


def morph(img, rng, strength=1.0, return_field=False):
    """Random scaling, shear, rotation, shift, elastic distortion and noise."""
    n = img.shape[0]
    c = (n - 1) / 2
    scale = np.exp(rng.uniform(-0.25, 0.25) * strength)    # about 0.78x .. 1.28x
    aspect = np.exp(rng.uniform(-0.15, 0.15) * strength)
    shear = rng.uniform(-0.3, 0.3) * strength
    angle = np.deg2rad(rng.uniform(-25, 25) * strength)
    shift = rng.uniform(-3, 3, size=2) * strength

    rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    shr = np.array([[1, shear], [0, 1]])
    scl = np.diag([scale * aspect, scale / aspect])
    A = np.linalg.inv(rot @ shr @ scl)                      # backward mapping

    y, x = np.mgrid[:n, :n].astype(float)
    p = np.stack([y - c - shift[0], x - c - shift[1]])
    src = np.tensordot(A, p, axes=1) + c

    fdy, fdx = random_field(n, max_disp=2.5 * strength, smooth=4, rng=rng)
    src[0] += fdy
    src[1] += fdx

    out = map_coordinates(img, src, order=1, mode="constant", cval=0.0)
    out = out + rng.normal(0, 0.05 * strength, out.shape)  # a little sensor noise
    out = np.clip(out, 0, 1).astype(np.float32)
    return (out, (fdy, fdx)) if return_field else out

In [ ]:
# One letter, its random vector field, and the result
demo_rng = np.random.default_rng(3)
src_img = templates[("B", "DejaVu Sans")]
morphed, (fdy, fdx) = morph(src_img, demo_rng, return_field=True)

ys, xs = np.mgrid[1:SIZE:2, 1:SIZE:2]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(src_img, cmap="gray"); axes[0].set_title("Template")
axes[1].imshow(src_img, cmap="gray", alpha=0.4)
axes[1].quiver(xs, ys, fdx[ys, xs], fdy[ys, xs], np.arctan2(fdy[ys, xs], fdx[ys, xs]),
               cmap="hsv", angles="xy", scale_units="xy", scale=0.5)
axes[1].set_title("Random elastic field (arrows 2x)")
axes[2].imshow(morphed, cmap="gray"); axes[2].set_title("Fully distorted")
for ax in axes:
    ax.axis("off")
plt.show()

# Many random versions of each letter
fig, axes = plt.subplots(3, 10, figsize=(14, 4.5))
for row, letter in enumerate(LETTERS):
    for col in range(10):
        font = FONTS[col % len(FONTS)]
        axes[row, col].imshow(morph(templates[(letter, font)], demo_rng), cmap="gray")
        axes[row, col].axis("off")
plt.suptitle("Random distortions at strength 1 (the training setting)")
plt.show()

## Part 3: Datasets

Because the images are generated, we can make as many as we like. Training, validation and test sets are generated independently, so the network is always tested on images it has never seen.

In [ ]:
def make_dataset(n, rng, strength=1.0):
    X = np.empty((n, SIZE, SIZE), np.float32)
    y = rng.integers(0, len(LETTERS), n)
    for i in range(n):
        font = FONTS[rng.integers(len(FONTS))]
        X[i] = morph(templates[(LETTERS[y[i]], font)], rng, strength)
    return X, y

X_train, y_train = make_dataset(6000, np.random.default_rng(1))
X_val,   y_val   = make_dataset(1500, np.random.default_rng(2))
X_test,  y_test  = make_dataset(1500, np.random.default_rng(3))

def to_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X).unsqueeze(1),   # shape (N, 1, 32, 32)
                       torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=64, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   shuffle=False)
print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("images per class (train):", dict(zip(LETTERS, np.bincount(y_train))))

## Part 4: A small CNN

Three blocks of *convolution → ReLU → max-pooling* turn the 32 × 32 image into 64 feature maps of 4 × 4 pixels. A small fully connected classifier then turns these into three scores, one per letter.

* **Convolutions** slide small learnable 3 × 3 filters over the image, detecting local patterns such as edges, corners and curves.
* **Max-pooling** halves the resolution, so later layers see larger parts of the letter and small shifts matter less.
* **Dropout** randomly switches off units during training, which reduces overfitting.

In [ ]:
class LetterCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # -> 16 x 16 x 16
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # -> 32 x 8 x 8
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # -> 64 x 4 x 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 4 * 4, 64), nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = LetterCNN().to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## Part 5: Training

In each epoch the network sees every training image once. For each batch it predicts the letters, measures the error with the **cross-entropy loss**, computes how each weight contributed to that error (**backpropagation**) and nudges the weights to reduce it (**Adam optimiser**). The validation set, never used for the weight updates, shows whether the network generalises.

In [ ]:
def run_epoch(model, loader, optimizer=None):
    """One pass over the data. Trains if an optimizer is given, otherwise only evaluates."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 10
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader)
    for key, val in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[key].append(val)
    print(f"epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"], "o-", label="validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(epochs, history["train_acc"], "o-", label="train")
axes[1].plot(epochs, history["val_acc"], "o-", label="validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.show()

## Part 6: Evaluation on the test set

The **confusion matrix** shows which letters get mixed up: rows are the true letters, columns the predictions. The diagonal holds the correct answers.

In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X).unsqueeze(1).to(device))
    return logits.argmax(1).cpu().numpy()

pred = predict(model, X_test)
print(f"Test accuracy: {(pred == y_test).mean():.3f}")

cm = np.zeros((len(LETTERS), len(LETTERS)), int)
np.add.at(cm, (y_test, pred), 1)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
for i in range(len(LETTERS)):
    for j in range(len(LETTERS)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(len(LETTERS)), LETTERS); ax.set_yticks(range(len(LETTERS)), LETTERS)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix")
plt.show()

In [ ]:
wrong = np.where(pred != y_test)[0]
print(f"{len(wrong)} of {len(y_test)} test images misclassified")
if len(wrong):
    show = wrong[:16]
    fig, axes = plt.subplots(1, len(show), figsize=(1.5 * len(show), 2))
    for ax, i in zip(np.atleast_1d(axes), show):
        ax.imshow(X_test[i], cmap="gray")
        ax.set_title(f"true {LETTERS[y_test[i]]}\npred {LETTERS[pred[i]]}", fontsize=8)
        ax.axis("off")
    plt.show()

## Part 7: Robustness to stronger distortions

The network was trained with distortion strength 1. How does it cope with weaker and stronger distortions? Accuracy usually stays high up to the training strength and then drops: a network mostly works within the range of variation it has seen. This is why **data augmentation**, training on deliberately varied images, matters so much.

In [ ]:
strengths = np.linspace(0, 2.5, 11)
accs = []
for k, s in enumerate(strengths):
    Xs, ys_ = make_dataset(1000, np.random.default_rng(100 + k), strength=s)
    accs.append((predict(model, Xs) == ys_).mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1.4]})
axes[0].plot(strengths, accs, "o-")
axes[0].axvline(1.0, color="gray", ls="--", label="training strength")
axes[0].set_xlabel("distortion strength"); axes[0].set_ylabel("accuracy")
axes[0].set_ylim(0, 1.02); axes[0].legend(); axes[0].set_title("Accuracy vs distortion")

examples = [morph(templates[("A", "DejaVu Sans")], np.random.default_rng(7), s) for s in strengths]
axes[1].imshow(np.hstack(examples), cmap="gray")
axes[1].set_xticks(np.arange(len(strengths)) * SIZE + SIZE / 2, [f"{s:.2g}" for s in strengths])
axes[1].set_yticks([]); axes[1].set_title("The same letter at each strength")
plt.show()

## Part 8: A look inside the network

**Filters:** the 16 learned 3 × 3 filters of the first layer. Many look like small edge or line detectors in different orientations.

**Feature maps:** what each first-layer filter responds to in one test image.

**Saliency:** how strongly the network's score for its chosen letter changes when each pixel changes (the gradient of the score with respect to the input). Bright pixels are the ones the decision depends on most.

In [ ]:
# First-layer filters
w = model.features[0].weight.detach().cpu().numpy()[:, 0]      # (16, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, f in zip(axes.flat, w):
    lim = np.abs(f).max()
    ax.imshow(f, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.axis("off")
plt.suptitle("First-layer filters (red = positive, blue = negative weights)")
plt.show()

# Feature maps of the first layer for one test image
x = torch.from_numpy(X_test[:1]).unsqueeze(1).to(device)
model.eval()
with torch.no_grad():
    maps = model.features[:2](x)[0].cpu().numpy()             # after conv + ReLU
fig, axes = plt.subplots(2, 9, figsize=(13, 3.2))
axes[0, 0].imshow(X_test[0], cmap="gray"); axes[0, 0].set_title("input")
axes[1, 0].axis("off")
for ax, m in zip(np.delete(axes.flat, [0, 9]), maps):
    ax.imshow(m, cmap="magma")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("First-layer feature maps")
plt.show()

In [ ]:
def saliency(model, img):
    x = torch.from_numpy(img[None, None]).to(device).requires_grad_(True)
    model.eval()
    scores = model(x)
    cls = scores.argmax(1).item()
    scores[0, cls].backward()
    return x.grad.abs()[0, 0].cpu().numpy(), cls

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for col in range(8):
    img = X_test[col]
    sal, cls = saliency(model, img)
    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"pred {LETTERS[cls]}")
    axes[1, col].imshow(img, cmap="gray")
    axes[1, col].imshow(sal, cmap="hot", alpha=0.6)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Top: test images.  Bottom: saliency (where the decision depends most)")
plt.show()

## Ideas to try

* **Train without augmentation:** call `make_dataset(..., strength=0)` for the training set only, retrain, and rerun Part 7. Accuracy on distorted test images collapses.
* **Train with stronger augmentation** (`strength=1.5`) and see how the robustness curve shifts.
* **Add more letters:** extend `LETTERS`, e.g. with `"D", "O", "Q"` (the pair O / Q is a nice hard case).
* **Shrink the network** (fewer filters) or train with fewer images, and watch the validation curve.
* **Use the lens distortions** from the earlier notebook as an extra augmentation.